# Clase 2 — MCP y Memoria Persistente

En esta clase llevamos el agente de la Clase 1 al siguiente nivel. Lo conectamos a servicios externos mediante **MCP** (Model Context Protocol) y resolvemos uno de los problemas más frecuentes en producción: que los agentes no recuerdan nada entre sesiones.

## Lo que vas a construir

Un agente de viajes corporativos que:
1. **Se conecta a un servidor MCP propio** que expone herramientas de búsqueda de vuelos y clima
2. **Usa los tres primitivos de MCP:** Tools, Resources y Prompts
3. **Persiste las preferencias del usuario** entre sesiones con mem0

## Arquitectura

```
Notebook (cliente MCP)
    └── MCPClient ──HTTP──► mcp_server.py
                                ├── Tool:     search_flights  (Duffel API)
                                ├── Tool:     get_offer_details
                                ├── Tool:     get_weather     (Open-Meteo)
                                ├── Resource: airports://list
                                └── Prompt:   planificar_viaje
```

> **⚠️ Prerequisito:** antes de correr cualquier celda, levantá el servidor MCP en una terminal separada:
> ```bash
> python mcp_server.py --http
> ```

## Setup

In [ ]:
import os
from dotenv import load_dotenv

# Carga las variables de entorno desde el archivo .env
# Necesitás dos claves en clase-2/.env:
#
#   DUFFEL_API_KEY=duffel_sandbox_...   → app.duffel.com (sandbox gratuito)
#   MEM0_API_KEY=m0-...                 → app.mem0.ai (plan gratuito disponible)
load_dotenv()

# Verificamos que ambas claves estén disponibles antes de avanzar
assert os.getenv("DUFFEL_API_KEY"), "Falta DUFFEL_API_KEY en .env — conseguila en app.duffel.com"
assert os.getenv("MEM0_API_KEY"),   "Falta MEM0_API_KEY en .env  — conseguila en app.mem0.ai"
print("✅ Setup OK")

---

## Parte 1 — MCP: Protocolo de Contexto del Modelo

**MCP** es el estándar abierto que permite a los agentes conectarse a cualquier servicio externo de forma uniforme. En lugar de escribir una función Python para cada API, empaquetamos las herramientas en un **servidor MCP** y el agente las descubre y las usa automáticamente.

### Los tres primitivos de MCP

| Primitivo | Para qué sirve | Ejemplo en este lab |
|-----------|---------------|---------------------|
| **Tool** | Ejecutar acciones con efecto | `search_flights`, `get_weather` |
| **Resource** | Exponer datos estáticos como contexto | `airports://list` |
| **Prompt** | Templates de instrucciones reutilizables | `planificar_viaje` |

### 1.1 Conectarse al servidor y usar Tools

In [ ]:
from strands import Agent
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp import MCPClient

# MCPClient necesita una función (lambda) que devuelva el transporte de conexión.
# Usamos streamable_http_client apuntando al servidor que levantamos con:
#   python mcp_server.py --http
mcp_client = MCPClient(lambda: streamable_http_client("http://localhost:8002/mcp"))

# El bloque `with` abre la conexión, la usa y la cierra automáticamente al salir.
# Es la forma más simple de conectarse, ideal para un uso puntual en una sola celda.
with mcp_client:
    # list_tools_sync() le pregunta al servidor qué herramientas expone.
    # El agente va a poder llamarlas exactamente igual que una @tool de Python.
    tools = mcp_client.list_tools_sync()
    print(f"🔧 Tools disponibles: {[t.tool_name for t in tools]}")

    # Le pasamos las tools al agente: a partir de acá puede usarlas cuando lo necesite.
    agente_demo = Agent(tools=[tools])
    agente_demo("Buscame vuelos de EZE a SFO para el 31 de mayo de 2026")

### 1.2 MCP Prompts — Templates de instrucciones reutilizables

Los **Prompts** son templates almacenados en el servidor. Permiten centralizar instrucciones complejas y reutilizarlas desde distintos agentes o aplicaciones, como si fuera un repositorio de prompts versionado. El cliente los instancia con parámetros concretos y el servidor devuelve el texto listo para usar.

In [ ]:
with mcp_client:
    # list_prompts_sync() devuelve los templates de prompts registrados en el servidor.
    prompts = mcp_client.list_prompts_sync()
    print("📝 Prompts disponibles:", [p.name for p in prompts.prompts])

    # get_prompt_sync() instancia el template llenando las variables con valores concretos.
    # El servidor genera el texto y lo devuelve listo para pasarle al agente.
    result = mcp_client.get_prompt_sync("planificar_viaje", {
        "origen": "Buenos Aires",
        "destino": "Miami",
        "fecha_ida": "2026-05-31"
    })
    prompt_text = result.messages[0].content.text
    print("📋 Prompt generado:\n", prompt_text)

    # El agente recibe el prompt ya armado como tarea inicial.
    # La ventaja: el prompt vive en el servidor, versionado y reutilizable.
    agente_demo = Agent(
        tools=[tools],
        system_prompt="Sos un asistente de viajes, responde siempre en español"
    )
    agente_demo(prompt_text)

### 1.3 Manejando el ciclo de vida de la conexión MCP

El bloque `with` es conveniente pero cierra la conexión al terminar. En un notebook necesitamos que la conexión quede abierta entre celdas para no reconectar cada vez. La solución es usar `start()` y `stop()` explícitos.

In [ ]:
# Problema del bloque `with`: la conexión se cierra al salir del bloque,
# por lo que no podemos reutilizarla en celdas posteriores del notebook.
#
# Solución: start() / stop() manuales. La conexión queda abierta hasta
# que llamemos a mcp_client.stop() — lo hacemos al final del notebook.

mcp_client = MCPClient(lambda: streamable_http_client("http://localhost:8002/mcp"))
mcp_client.start()  # ← abre la conexión; recordar cerrarla con mcp_client.stop()

# Ahora podemos crear múltiples agentes y ejecutar celdas sin reconectar.
agent_session_1 = Agent(
    tools=mcp_client.list_tools_sync(),
    system_prompt="Sos un asistente de viajes corporativos. Responde en español.",
)
print("✅ Conexión MCP iniciada — agente listo")

---

## Parte 2 — El problema de la memoria en agentes

La "memoria" de un agente Strands es simplemente su lista `messages[]`. Cuando creamos una nueva instancia de `Agent`, esa lista empieza **vacía** — el agente no sabe nada de conversaciones anteriores.

Esto es un problema cuando:
- El mismo usuario vuelve en una sesión nueva (al día siguiente, por ejemplo)
- Queremos que el agente recuerde preferencias a largo plazo
- Múltiples instancias del agente necesitan compartir contexto

Vamos a demostrarlo en dos pasos:

### 2.1 Primera sesión — guardando preferencias en el historial

In [ ]:
# Primera sesión: el usuario comparte sus preferencias.
# El agente las guarda en agent_session_1.messages (lista en RAM).
# Este historial es local a esta instancia y no sobrevive si la reiniciamos.
response = agent_session_1("Siempre prefiero asiento de pasillo y soy vegetariano. Recordalo.")

### 2.2 Nueva sesión — el agente no recuerda nada

Creamos una nueva instancia de `Agent`. Simula el escenario de un usuario que vuelve al día siguiente. Aunque el servidor MCP es el mismo, el historial de mensajes no persiste.

In [ ]:
# Nueva instancia = historial vacío. Simula al usuario que vuelve en una sesión nueva.
# Aunque usamos las mismas tools del mismo servidor MCP, este agente no tiene acceso
# a los mensajes de agent_session_1 — porque la "memoria" de Strands es solo messages[].
agent_session_2 = Agent(
    tools=mcp_client.list_tools_sync(),
    system_prompt="Sos un asistente de viajes corporativos. Responde en español.",
)
# ⚠️ Resultado esperado: el agente responde que no tiene ninguna preferencia cargada.
response = agent_session_2("¿Que preferencias tengo cargadas?")

---

## Parte 3 — Memoria Persistente con mem0

**mem0** es una capa de memoria de largo plazo para agentes. Almacena hechos en una base de datos vectorial y los recupera por relevancia semántica. A diferencia del historial `messages[]`, la memoria sobrevive entre sesiones.

### Comparación

| | Historial `messages[]` | mem0 |
|---|---|---|
| **Duración** | Una sesión (en RAM) | Permanente (base de datos) |
| **Recuperación** | Todo el historial | Búsqueda semántica |
| **Compartido** | No — por instancia | Sí — por `user_id` |
| **Límite** | Ventana de contexto del modelo | Millones de memorias |

> **🔑 Requisito:** para usar mem0 necesitás una API key de [app.mem0.ai](https://app.mem0.ai).
> Registrate, copiá la key y agregala en `clase-2/.env`:
> ```
> MEM0_API_KEY=m0-...
> ```
> El plan gratuito es suficiente para este lab.

### 3.1 Configurar el system prompt para usar mem0

El agente usa `mem0_memory` como cualquier otra tool. El system prompt le indica cuándo guardar (`action='store'`) y cuándo recuperar (`action='retrieve'`) información del usuario.

In [ ]:
from strands_tools import mem0_memory

# System prompt con instrucciones explícitas para usar mem0_memory.
# Le indicamos al agente cuándo guardar (action='store') y cuándo recuperar
# (action='retrieve') preferencias del usuario. El agente decide cuándo llamar
# a la tool igual que con cualquier otra — nosotros solo le damos las reglas.
SYSTEM_PROMPT = """
Sos un asistente de viajes corporativos para la empresa ACME Corp.

Reglas:
- Al inicio de cada conversacion, usa mem0_memory con action='retrieve' para cargar las preferencias del usuario.
- Cuando el usuario comparta una preferencia, usa mem0_memory con action='store' para guardarla.
- Busca vuelos con search_flights y clima con get_weather cuando te lo pidan.
- Responde siempre en español.
- Se conciso y profesional.
"""
print("✅ mem0_memory importado — SYSTEM_PROMPT definido")

### 3.2 Combinar MCP tools con mem0 y guardar preferencias

Creamos el agente final combinando las tools del servidor MCP con `mem0_memory`. En los logs de la ejecución vas a poder ver cómo el agente llama a `mem0_memory` con `action='store'` de forma autónoma.

In [ ]:
# Combinamos las tools del servidor MCP con mem0_memory de strands_tools.
# Strands acepta una lista de listas de tools — las aplana internamente.
mcp_tools = mcp_client.list_tools_sync()
all_tools = [mcp_tools, mem0_memory]

# Creamos el agente con el system prompt que le indica cómo usar mem0.
agent = Agent(tools=all_tools, system_prompt=SYSTEM_PROMPT)

# USER_ID identifica al usuario en mem0. Las memorias son por usuario,
# así que distintos valores de USER_ID tienen memorias separadas.
USER_ID = 'ricardocursostrands2026'

# Primera interacción: el agente guarda las preferencias en mem0 (base de datos vectorial).
# Observá en los logs cómo llama a mem0_memory con action='store' automáticamente.
response = agent(
    f"Hola, soy {USER_ID}. Siempre prefiero asiento de pasillo, "
    f"vuelo por American Airlines cuando puedo, y soy vegetariano. "
    f"Acordate de todo esto."
)

### 3.3 Buscar vuelos con preferencias del usuario en contexto

El agente ya tiene las preferencias en su historial de esta sesión. Observá cómo las aplica al recomendar opciones de vuelo.

In [ ]:
# Segunda interacción en la misma sesión.
# El agente ya tiene las preferencias en messages[] y en mem0.
# Observá cómo las aplica al recomendar vuelos: aerolínea, opciones de comida, etc.
agent("Buscame vuelos de EZE a SFO para el 30 de mayo de 2026")

### 3.4 Persistencia entre sesiones — la demostración clave

Creamos una **nueva instancia de `Agent`** — igual que en la Parte 2, el historial `messages[]` empieza vacío. Pero esta vez el system prompt le indica al agente que recupere las preferencias de mem0 al inicio de la conversación.

**Resultado esperado:** el agente conoce asiento de pasillo, American Airlines y dieta vegetariana sin que el usuario se lo haya dicho en esta sesión.

In [ ]:
# Nueva instancia — igual que en la Parte 2, el historial messages[] empieza vacío.
# Pero ahora el system_prompt le indica al agente que llame a mem0_memory con
# action='retrieve' al inicio, recuperando las preferencias de la sesión anterior.
agent_v2 = Agent(
    tools=all_tools,
    system_prompt=SYSTEM_PROMPT,
)

# 🎯 Resultado esperado: el agente recupera asiento de pasillo, American Airlines
# y dieta vegetariana — aunque nunca haya conversado con este usuario antes.
response = agent_v2(f"Soy {USER_ID}. Decime cuales son mis preferencias")

# Cerramos la conexión MCP al terminar el notebook.
mcp_client.stop()
print("✅ Conexión MCP cerrada")

---

## ✅ Resumen de la Clase 2

### Lo que aprendiste

| Concepto | Clave |
|----------|-------|
| **MCP Tools** | El agente llama a herramientas de un servidor externo igual que a una `@tool` de Python |
| **MCP Resources** | Datos estáticos que el servidor expone como contexto (IATA, catálogos, etc.) |
| **MCP Prompts** | Templates versionados en el servidor; el cliente los instancia con parámetros |
| **`with` vs `start()`** | `with` para uso puntual; `start()/stop()` para mantener la conexión entre celdas |
| **Aislamiento de memoria** | Cada instancia de `Agent` tiene su propio `messages[]` — no hay memoria compartida por defecto |
| **mem0** | Capa de memoria persistente por `user_id`; sobrevive entre sesiones y entre instancias |

### Próximos pasos

En la **Clase 3** vamos a ver cómo llevar este agente a producción: empaquetarlo como API, agregarle observabilidad y manejar errores en escenarios reales.